In [ ]:
!pip install torch==2.0.1 --index-url https://download.pytorch.org/whl/cpu


Looking in indexes: https://download.pytorch.org/whl/cpu
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 195.4/195.4 MB 4.8 MB/s eta 0:00:00
  Attempting uninstall: torch
    Found existing installation: torch 2.6.0+cu124
    Uninstalling torch-2.6.0+cu124:
      Successfully uninstalled torch-2.6.0+cu124
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchaudio 2.6.0+cu124 requires torch==2.6.0, but you have torch 2.0.1+cpu which is incompatible.
torchvision 0.21.0+cu124 requires torch==2.6.0, but you have torch 2.0.1+cpu which is incompatible.


In [ ]:
!pip install torch numpy pandas scikit-learn rtdl --quiet


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.4/887.4 MB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.1/317.1 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.0/21.0 MB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.3/849.3 kB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.1/557.1 MB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 44.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchdata 0.11.0 requires torch>=2, but you have torch 1.13.1 which is incompatible.
torchaudio 2.6.0+cu124 requires torch==2.6.0, but you have torch 1.13.1 which is incompatible.
accelerate 1.7.0 requires torch>=2.0.0, but you have torch 1.13.1 which is incompatibl

In [ ]:
!pip install --upgrade --force-reinstall numpy==1.23.5
# (Ví dụ: 1.23.5 chỉ là một ví dụ; bạn có thể chọn phiên bản phù hợp với sklearn, torch, rtdl trong môi trường của bạn)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 87.1 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
treescope 0.1.9 requires numpy>=1.25.2, but you have numpy 1.23.5 which is incompatible.
chex 0.1.89 requires numpy>=1.24.1, but you have numpy 1.23.5 which is incompatible.
scikit-image 0.25.2 requires numpy>=1.24, but you have numpy 1.23.5 which is incompatible.
blosc2 3.3.4 requires numpy>=1.26, but you have numpy 1.23.5 which is incompatible.
jax 0.5.2 requires numpy>=1.25, but you have numpy 1.23.5 which is incompatible.
tensorflow 2.18.0 requires numpy<2.1.0,>=1.26.0, but you have numpy 1.23.5 which is incompatible.
xarray 2025.3.1 requires numpy>=1.24, but you have numpy 1.23.5 whi

In [ ]:
import pandas as pd
import numpy as np

# Đọc 4 file
red = pd.read_csv("../Data/Red.csv")
white = pd.read_csv("../Data/White.csv")
rose = pd.read_csv("../Data/Rose.csv")
sparkling = pd.read_csv("../Data/Sparkling.csv")

# Gán nhãn loại rượu
red["WineStyle"] = "Red"
white["WineStyle"] = "White"
rose["WineStyle"] = "Rose"
sparkling["WineStyle"] = "Sparkling"

# Gộp toàn bộ
df = pd.concat([red, white, rose, sparkling], ignore_index=True)

# Loại bỏ các dòng Year không hợp lệ
df = df[df["Year"] != "N.V."].copy()
df["Year"] = df["Year"].astype(int)
df["Age"] = 2025 - df["Year"]

# Placeholder cho Variety nếu thiếu
df["Variety"] = df.get("Variety", "Unknown")

# Đặc trưng chuẩn hóa mở rộng
df["PricePerAge"] = df["Price"] / (df["Age"] + 1)
df["LogPrice"] = np.log1p(df["Price"])
df["LogLogPrice"] = np.log1p(df["LogPrice"])
df["PriceAgeInteraction"] = df["PricePerAge"] * df["LogPrice"]
df["IsOld"] = df["Age"] >= 10
df["LogNumberOfRatings"] = np.log1p(df["NumberOfRatings"])
df["RatingSupportScore"] = df["NumberOfRatings"] / (df["Age"] + 1)

# Smoothed rating
global_mean = df["Rating"].mean()
df["SmoothedRating"] = (df["Rating"] * df["NumberOfRatings"] + global_mean * 5) / (df["NumberOfRatings"] + 5)

# Chia nhóm độ tin cậy
def reliability_group(n):
    if n < 35: return "Low"
    elif n < 838: return "Medium"
    else: return "High"
df["ReliabilityGroup"] = df["NumberOfRatings"].apply(reliability_group)

# Các đặc trưng thống kê theo nhóm
df["Country_Rank"] = df["Country"].map(df.groupby("Country")["SmoothedRating"].mean().rank(ascending=False).astype(int))
df["Region_Rank"] = df["Region"].map(df.groupby("Region")["SmoothedRating"].mean().rank(ascending=False).astype(int))
winery_mean = df.groupby("Winery")["SmoothedRating"].mean()
df["Winery_Score"] = df["Winery"].map((winery_mean.max() - winery_mean) / (winery_mean.max() - 1))
df["Variety_Freq"] = df["Variety"].map(df["Variety"].value_counts(normalize=True))

print("✅ Đã xử lý đặc trưng. Số dòng còn lại:", len(df))


✅ Đã xử lý đặc trưng. Số dòng còn lại: 13090


In [ ]:
from sklearn.model_selection import train_test_split

# Tách dữ liệu theo nhóm độ tin cậy
low_df    = df[df["ReliabilityGroup"] == "Low"].copy()
mid_df    = df[df["ReliabilityGroup"] == "Medium"].copy()
high_df   = df[df["ReliabilityGroup"] == "High"].copy()

# Tách train/test theo nhóm (tách test cố định 20%)
low_train,  low_test  = train_test_split(low_df,  test_size=0.2, random_state=42)
mid_train,  mid_test  = train_test_split(mid_df,  test_size=0.2, random_state=42)
high_train, high_test = train_test_split(high_df, test_size=0.2, random_state=42)

# Gộp lại các tập huấn luyện hỗ trợ (giống file LightGBM_ByGroup)
low_train_blended  = pd.concat([low_train, mid_train.sample(frac=0.3, random_state=42)], ignore_index=True)
high_train_blended = pd.concat([high_train, mid_train.sample(frac=0.3, random_state=42)], ignore_index=True)

# ✅ In số lượng để kiểm tra
print("Low  train:", len(low_train_blended), "| test:", len(low_test))
print("Mid  train:", len(mid_train),         "| test:", len(mid_test))
print("High train:", len(high_train_blended), "| test:", len(high_test))


Low  train: 3617 | test: 268
Mid  train: 8497 | test: 2125
High train: 3454 | test: 227


In [ ]:
from sklearn.preprocessing import StandardScaler, OrdinalEncoder

# ===== CÁC CỘT SỬ DỤNG CHUNG CHO TẤT CẢ CÁC NHÓM =====
target = "Rating"
features = [
    "Country", "Region", "Winery", "WineStyle", "Variety",  # categorical
    "Price", "Age", "PricePerAge", "LogPrice", "LogLogPrice",
    "PriceAgeInteraction", "IsOld", "LogNumberOfRatings",
    "RatingSupportScore", "Country_Rank", "Region_Rank",
    "Winery_Score", "Variety_Freq"
]

# Phân loại categorical và numerical
categorical_cols = ["Country", "Region", "Winery", "WineStyle", "Variety"]
numerical_cols = [f for f in features if f not in categorical_cols]

# ===== DEFINE HÀM XỬ LÝ =====
def encode_and_scale(train_df, test_df):
    # 1. Encode categorical
    enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    train_cat = enc.fit_transform(train_df[categorical_cols])
    test_cat  = enc.transform(test_df[categorical_cols])

    # 2. Scale numerical
    scaler = StandardScaler()
    train_num = scaler.fit_transform(train_df[numerical_cols])
    test_num  = scaler.transform(test_df[numerical_cols])

    # 3. Gộp lại
    X_train = np.hstack([train_cat, train_num])
    X_test  = np.hstack([test_cat, test_num])

    y_train = train_df[target].to_numpy().reshape(-1, 1)
    y_test  = test_df[target].to_numpy().reshape(-1, 1)

    return X_train, y_train, X_test, y_test

# ===== ÁP DỤNG CHO 3 NHÓM =====
low_X_train,  low_y_train,  low_X_test,  low_y_test   = encode_and_scale(low_train_blended, low_test)
mid_X_train,  mid_y_train,  mid_X_test,  mid_y_test   = encode_and_scale(mid_train, mid_test)
high_X_train, high_y_train, high_X_test, high_y_test = encode_and_scale(high_train_blended, high_test)

print("✅ Đã encode và scale xong cho cả 3 nhóm.")


✅ Đã encode và scale xong cho cả 3 nhóm.


In [ ]:
import torch
import torch.nn as nn
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import rtdl

# Thiết bị huấn luyện
device = "cuda" if torch.cuda.is_available() else "cpu"

# Hàm huấn luyện và đánh giá 1 mô hình RTDL-ResNet
def train_rtdl_resnet(X_train, y_train, X_test, y_test, group_name):
    # Chuyển về tensor
    X_train_tensor = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_train_tensor = torch.tensor(y_train, dtype=torch.float32).to(device)
    X_test_tensor  = torch.tensor(X_test, dtype=torch.float32).to(device)
    y_test_tensor  = torch.tensor(y_test, dtype=torch.float32).to(device)

    # Định nghĩa mô hình
    model = rtdl.ResNet(
        d_in=X_train.shape[1],
        d_out=1,
        n_blocks=4,
        d_main=256,
        d_hidden=128,
        dropout_first=0.2,
        dropout_second=0.1,
        normalization="BatchNorm1d",
        activation="ReLU"
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
    loss_fn = nn.MSELoss()

    # Huấn luyện 300 epoch (có thể tăng nếu muốn)
    for epoch in range(1, 301):
        model.train()
        optimizer.zero_grad()
        y_pred = model(X_train_tensor)
        loss = loss_fn(y_pred, y_train_tensor)
        loss.backward()
        optimizer.step()

    # Đánh giá
    model.eval()
    with torch.no_grad():
        y_pred = model(X_test_tensor).cpu().numpy()
        y_true = y_test_tensor.cpu().numpy()


In [ ]:
import torch
import torch.nn as nn
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import rtdl

# Thiết bị huấn luyện
device = "cuda" if torch.cuda.is_available() else "cpu"

# Hàm huấn luyện và đánh giá 1 mô hình RTDL-ResNet
def train_rtdl_resnet(X_train, y_train, X_test, y_test, group_name):
    # Chuyển về tensor
    X_train_tensor = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_train_tensor = torch.tensor(y_train, dtype=torch.float32).to(device)
    X_test_tensor  = torch.tensor(X_test, dtype=torch.float32).to(device)
    y_test_tensor  = torch.tensor(y_test, dtype=torch.float32).to(device)

    # Định nghĩa mô hình
    model = rtdl.ResNet(
        d_in=X_train.shape[1],
        d_out=1,
        n_blocks=4,
        d_main=256,
        d_hidden=128,
        dropout_first=0.2,
        dropout_second=0.1,
        normalization="BatchNorm1d",
        activation="ReLU"
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
    loss_fn = nn.MSELoss()

    # Huấn luyện 300 epoch (có thể tăng nếu muốn)
    for epoch in range(1, 1501):
        model.train()
        optimizer.zero_grad()
        y_pred = model(X_train_tensor)
        loss = loss_fn(y_pred, y_train_tensor)
        loss.backward()
        optimizer.step()
        if r2 > best_r2:
            best_r2 = r2
            counter = 0
        else:
            counter += 1

        if counter >= patience:
            break

    # Đánh giá
    model.eval()
    with torch.no_grad():
        y_pred = model(X_test_tensor).cpu().numpy()
        y_true = y_test_tensor.cpu().numpy()


In [ ]:
# Đầu vào: 1 dòng đã được encode + có ReliabilityGroup
# Đầu ra: prediction theo chiến lược
def smart_predict(row, models, encoders, features, group):
    # Lấy nhóm độ tin cậy
    group_name = row["ReliabilityGroup"]

    # Tách đặc trưng đầu vào
    row_features = row[features].copy().to_frame().T

    # Encode và scale lại
    cat_cols = encoders["cat_cols"]
    num_cols = encoders["num_cols"]

    cat = encoders["cat_enc"].transform(row_features[cat_cols])
    num = encoders["num_scaler"].transform(row_features[num_cols])
    x_input = np.hstack([cat, num])
    x_tensor = torch.tensor(x_input, dtype=torch.float32).to(device)

    # Với nhóm Medium → dùng model_mid_only
    if group_name == "Medium":
        y_pred = models["mid"].eval()(x_tensor).cpu().item()
    elif group_name == "Low":
        y_pred_low = models["low"].eval()(x_tensor).cpu().item()
        y_pred_mid = models["mid"].eval()(x_tensor).cpu().item()
        y_pred = (y_pred_low + y_pred_mid) / 2
    elif group_name == "High":
        y_pred_high = models["high"].eval()(x_tensor).cpu().item()
        y_pred_mid  = models["mid"].eval()(x_tensor).cpu().item()
        y_pred = (y_pred_high + y_pred_mid) / 2
    else:
        y_pred = models["mid"].eval()(x_tensor).cpu().item()

    return y_pred


In [ ]:
def train_and_evaluate(X_train, y_train, X_test, y_test, group_name):
    # Convert to tensor

    import torch
    import torch.nn as nn
    import rtdl
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
    import os

    # Device: GPU nếu có, ngược lại CPU
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    X_train_tensor = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_train_tensor = torch.tensor(y_train, dtype=torch.float32).to(device)
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
    y_test_tensor = torch.tensor(y_test, dtype=torch.float32).to(device)

    # Model
    model = rtdl.ResNet(
        d_in=X_train.shape[1], d_out=1,
        n_blocks=4, d_main=256, d_hidden=128,
        dropout_first=0.2, dropout_second=0.1,
        normalization="BatchNorm1d", activation="ReLU"
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
    loss_fn = nn.MSELoss()

    # Train
    for epoch in range(1, 301):
        model.train()
        optimizer.zero_grad()
        y_pred = model(X_train_tensor)
        loss = loss_fn(y_pred, y_train_tensor)
        loss.backward()
        optimizer.step()


    os.makedirs("../models", exist_ok=True)

    #    - Tạo một dummy_input với đúng số chiều đầu vào (batch_size=1, d_in)
    d_in = X_train.shape[1]
    dummy_input = torch.randn(1, d_in, dtype=torch.float32).to(device)

    #    - Đặt tên file ONNX theo nhóm (Low/Mid/High)
    onnx_filename = f"../models/ResNet_{group_name}.onnx"
    torch.onnx.export(
        model,
        dummy_input,
        onnx_filename,
        input_names=["input"],
        output_names=["output"],
        dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}},
        opset_version=11
    )
    print(f"✅ Đã xuất mô hình nhóm {group_name} sang {onnx_filename}")

    # Evaluate
    def get_metrics(y_true, y_pred):
        mae = mean_absolute_error(y_true, y_pred)
        mse = mean_squared_error(y_true, y_pred)
        rmse = mse ** 0.5
        r2 = r2_score(y_true, y_pred)
        return mae, rmse, r2


    model.eval()
    with torch.no_grad():
        pred_train = model(X_train_tensor).cpu().numpy()
        pred_test  = model(X_test_tensor).cpu().numpy()
        y_train_np = y_train_tensor.cpu().numpy()
        y_test_np  = y_test_tensor.cpu().numpy()

        train_metrics = get_metrics(y_train_np, pred_train)
        test_metrics  = get_metrics(y_test_np, pred_test)

    return train_metrics, test_metrics



In [ ]:
groups = {
    "Low":   (low_X_train, low_y_train, low_X_test, low_y_test),
    "Medium":(mid_X_train, mid_y_train, mid_X_test, mid_y_test),
    "High":  (high_X_train, high_y_train, high_X_test, high_y_test),
}

for name, (X_tr, y_tr, X_te, y_te) in groups.items():
    train_scores, test_scores = train_and_evaluate(X_tr, y_tr, X_te, y_te, name)
    print(f"\n📊 {name} group:")
    print(f"  Train - MAE: {train_scores[0]:.4f}, RMSE: {train_scores[1]:.4f}, R²: {train_scores[2]:.4f}")
    print(f"  Test  - MAE: {test_scores[0]:.4f}, RMSE: {test_scores[1]:.4f}, R²: {test_scores[2]:.4f}")


✅ Đã xuất mô hình nhóm Low sang models/ResNet_Low.onnx

📊 Low group:
  Train - MAE: 0.1417, RMSE: 0.1829, R²: 0.6402
  Test  - MAE: 0.1557, RMSE: 0.1962, R²: 0.6194
✅ Đã xuất mô hình nhóm Medium sang models/ResNet_Medium.onnx

📊 Medium group:
  Train - MAE: 0.1340, RMSE: 0.1717, R²: 0.6645
  Test  - MAE: 0.1405, RMSE: 0.1777, R²: 0.6304
✅ Đã xuất mô hình nhóm High sang models/ResNet_High.onnx

📊 High group:
  Train - MAE: 0.1268, RMSE: 0.1621, R²: 0.7094
  Test  - MAE: 0.1264, RMSE: 0.1598, R²: 0.6944
